In [5]:
from PIL import Image
import os

def GetBBox(bbox, imgWidth, imgHeight, paddingPercent=0.1):
    x_min, y_min, x_max, y_max = bbox
    box_width = x_max - x_min
    box_height = y_max - y_min

    #Padding to save information about skin and optimize segmentation
    pad_x = box_width * paddingPercent
    pad_y = box_height * paddingPercent


    new_x_min = int(max(0, x_min - pad_x))
    new_y_min = int(max(0, y_min - pad_y))
    new_x_max = int(min(imgWidth, x_max + pad_x))
    new_y_max = int(min(imgHeight, y_max + pad_y))

    return new_x_min, new_y_min, new_x_max, new_y_max


def CreateDataset(maskPath, outputPath):
    for maskName in os.listdir(maskPath):
        mask=Image.open(os.path.join(maskPath, maskName))
        bbox = mask.getbbox()
        width, height = mask.size
        x_min, y_min, x_max, y_max=GetBBox(bbox, width, height)
        croppedMask = mask.crop((x_min, y_min, x_max, y_max))
        base_name, _ = os.path.splitext(maskName)
        newName = base_name + ".png"
        save_path = os.path.join(outputPath, newName)
        croppedMask.save(save_path)

In [ ]:
# CreateDataset("dataset/val/valGt", "SegDataset/val/valGt")

In [3]:
from PIL import Image
import os
def CropOnlyImages(imgSrcDir, originalMaskSrcDir, imgOutDir, paddingPercent=0.1):

    for imgName in os.listdir(imgSrcDir):
        # Przetwarzamy tylko pliki graficzne
        if not imgName.lower().endswith(('.png', '.jpg', '.jpeg', '.tif')):
            continue

        base_name = os.path.splitext(imgName)[0]
        maskName = f"{base_name}_segmentation.png"
        
        imgPath = os.path.join(imgSrcDir, imgName)
        # WAŻNE: To musi być ścieżka do folderu z NIEPRZYCIĘTYMI maskami
        maskPath = os.path.join(originalMaskSrcDir, maskName)

        if not os.path.exists(maskPath):
            print(f"Pominięto: Brak oryginalnej maski dla obrazu {imgName}")
            continue

        img = Image.open(imgPath).convert("RGB")
        mask = Image.open(maskPath).convert("L")

        # Pobieramy bounding box z oryginalnej maski
        bbox = mask.getbbox()
        if bbox is None:
            print(f"Maska {maskName} jest pusta, pomijam.")
            continue

        width, height = mask.size 
        
        # Wyliczamy współrzędne
        x_min, y_min, x_max, y_max = GetBBox(bbox, width, height, paddingPercent)
        
        # Docinamy tylko obraz
        croppedImg = img.crop((x_min, y_min, x_max, y_max))
        
        # Zapisujemy docięty obraz
        croppedImg.save(os.path.join(imgOutDir, imgName))
        print(f"Pomyślnie przycięto obraz: {imgName}")

In [ ]:
# CropOnlyImages("dataset/val/valInput", "dataset/val/valGt", "SegDataset/val/valInput")

Pomyślnie przycięto obraz: ISIC_0000092.jpg
Pomyślnie przycięto obraz: ISIC_0013222.jpg
Pomyślnie przycięto obraz: ISIC_0015079.jpg
Pomyślnie przycięto obraz: ISIC_0000124.jpg
Pomyślnie przycięto obraz: ISIC_0013128.jpg
Pomyślnie przycięto obraz: ISIC_0000443.jpg
Pomyślnie przycięto obraz: ISIC_0000225.jpg
Pomyślnie przycięto obraz: ISIC_0010467.jpg
Pomyślnie przycięto obraz: ISIC_0000008.jpg
Pomyślnie przycięto obraz: ISIC_0010213.jpg
Pomyślnie przycięto obraz: ISIC_0000258.jpg
Pomyślnie przycięto obraz: ISIC_0009958.jpg
Pomyślnie przycięto obraz: ISIC_0000251.jpg
Pomyślnie przycięto obraz: ISIC_0011164.jpg
Pomyślnie przycięto obraz: ISIC_0010552.jpg
Pomyślnie przycięto obraz: ISIC_0010553.jpg
Pomyślnie przycięto obraz: ISIC_0012432.jpg
Pomyślnie przycięto obraz: ISIC_0013333.jpg
Pomyślnie przycięto obraz: ISIC_0012338.jpg
Pomyślnie przycięto obraz: ISIC_0016064.jpg
Pomyślnie przycięto obraz: ISIC_0013644.jpg
Pomyślnie przycięto obraz: ISIC_0010046.jpg
Pomyślnie przycięto obraz: ISIC_

In [12]:
counter=0
for file in os.listdir("SegDataset/val/valInput"):
    img=Image.open(os.path.join("SegDataset/val/valInput", file))
    width, height = img.size
    if width < 256 or height < 256:
            counter += 1
print(f"There are {counter} images with size less than 256")

There are 5 images with size less than 256


In [16]:
counter=0
minWidth=0
minHeight=0
for file in os.listdir("SegDataset/train/trainInput"):
    img=Image.open(os.path.join("SegDataset/train/trainInput", file))
    width, height = img.size
    if width < 256 or height < 256:
            counter += 1
            minWidth=width
            minHeight=height
print(f"There are {counter} images with size less than 256")
print(minWidth)
print(minHeight)

There are 37 images with size less than 256
255
207


In [15]:
counter=0
for file in os.listdir("SegDataset/test/testInput"):
    img=Image.open(os.path.join("SegDataset/test/testInput", file))
    width, height = img.size
    if width < 256 or height < 256:
            counter += 1
print(f"There are {counter} images with size less than 256")

There are 5 images with size less than 256
